# Tutorial 10: MCP Authentication for Microsoft Foundry Agents

## Why This Matters for Enterprise AI

When building enterprise AI agents, **security and proper authentication are non-negotiable**. Agents need to:
- Securely access internal APIs and databases
- Act on behalf of users without exposing credentials
- Integrate properly with SaaS tools (GitHub, Salesforce, M365)
- Maintain audit trails for compliance (SOC2, HIPAA, etc.)

**Model Context Protocol (MCP)** provides a standardized way for AI agents to call external tools. But in the enterprise, you can't just call APIs — you need proper authentication.

**This tutorial shows how to securely connect Foundry agents to MCP servers using 4 authentication patterns.**

---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  Microsoft Foundry MCP Authentication Architecture                          │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌──────────────────┐                                                       │
│  │   User/Client    │                                                       │
│  └────────┬─────────┘                                                       │
│           │                                                                 │
│           ▼                                                                 │
│  ┌──────────────────────────────────────────────────────────────────────┐  │
│  │  Microsoft Foundry Project                                           │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐ │  │
│  │  │  Agent Identity                                                 │ │  │
│  │  │  • Shared Project ID (unpublished agents)                       │ │  │
│  │  │  • Individual Agent ID (published agents)                       │ │  │
│  │  └─────────────────────────────────────────────────────────────────┘ │  │
│  │                                                                      │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐ │  │
│  │  │  Project Connections (Credential Store)                         │ │  │
│  │  │  • API Keys, PAT Tokens                                         │ │  │
│  │  │  • OAuth Client Credentials                                     │ │  │
│  │  │  • Agentic Identity Token Config                                │ │  │
│  │  └─────────────────────────────────────────────────────────────────┘ │  │
│  │                                                                      │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐ │  │
│  │  │  Foundry Agent Service                                          │ │  │
│  │  │  • Create agents with MCP tools                                 │ │  │
│  │  │  • Manage approval workflows                                    │ │  │
│  │  │  • Handle OAuth consent flows                                   │ │  │
│  │  └─────────────────────────────────────────────────────────────────┘ │  │
│  └──────────────────────────────────────────────────────────────────────┘  │
│           │                                                                 │
│           │ Auth Method Selection                                           │
│           ▼                                                                 │
│  ┌───────────────────────────────────────────────────────────────────────┐ │
│  │                     MCP Server                                        │ │
│  │  ┌─────────────┐ ┌─────────────┐ ┌─────────────┐ ┌─────────────────┐ │ │
│  │  │ Unauth      │ │ Key-based   │ │ Agentic ID  │ │ OAuth Passthru  │ │ │
│  │  │ (Public)    │ │ (API Key)   │ │ (MI Token)  │ │ (User Context)  │ │ │
│  │  └─────────────┘ └─────────────┘ └─────────────┘ └─────────────────┘ │ │
│  └───────────────────────────────────────────────────────────────────────┘ │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Key Documentation Links

| Resource | Description | URL |
|----------|-------------|-----|
| **MCP Authentication** | Official authentication patterns for MCP in Foundry | [learn.microsoft.com](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/mcp-authentication?view=foundry) |
| **Agent Identity Concepts** | Understanding agent identities in Entra ID | [learn.microsoft.com](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/agent-identity?view=foundry) |
| **Azure AI Projects SDK** | Python SDK samples for agents | [GitHub](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects/samples) |
| **Azure AI Agents SDK** | Agent-specific SDK with MCP samples | [GitHub](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-agents/samples) |
| **Microsoft Foundry Portal** | Create and manage agents | [ai.azure.com](https://ai.azure.com) |
| **Entra Agent Identities** | Manage agent identities in Entra | [entra.microsoft.com](https://entra.microsoft.com/#view/Microsoft_AAD_RegisteredApps/AllAgents.MenuView) |

## Part 1: Understanding Agent Identity in Microsoft Foundry

### The Enterprise Challenge: "Who Is This Agent?"

Imagine an AI agent booking conference rooms, approving purchase orders, or accessing customer data. The security team asks: *"How do we audit what the agent did vs. what a human did?"*

Traditional service principals don't distinguish between bots and humans. **Agent Identity** solves this.

### What Is Agent Identity?

**Agent Identity** is a special service principal in Microsoft Entra ID designed specifically for AI agents. Think of it as giving your AI agent its own "employee ID badge" within your organization.

| Feature | Why It Matters for Enterprise |
|-----------|------------------------------|
| **Identified Actions** | Auditors can filter logs to see "what the agent did" vs. "what a human did" |
| **Right-Sized Access** | Grant agent-specific permissions (e.g., "can read but not delete") |
| **Security Controls** | Prevent agents from granting themselves admin access or escalating privileges |
| **Scalable Management** | Spin up/down hundreds of agents without cluttering the directory |

### Shared vs Individual Agent Identity

| ID Type | When to Use | Enterprise Example |
|--------------|-------------|--------------------|
| **Shared Project ID** | Development/testing phase | All agents in the "HR Assistant" project share one ID during development |
| **Individual Agent ID** | Production deployment | "Sales Copilot" gets its own ID with access only to CRM data |

**Enterprise Recommendation**: Always use **individual IDs** in production. This enables:
- Per-agent audit trails ("Which agent accessed this customer record?")
- Granular permission revocation (disable one agent without affecting others)
- Compliance-friendly separation of concerns

### Agent Identity Blueprint

**Agent Identity Blueprint** is a template that governs how agent identities are created:

- **Type Classification**: Categorizes agents (e.g., "Contoso Sales Agent")
- **ID Creation Permissions**: Defines OAuth credentials for identity management
- **Runtime Authentication**: Used to acquire access tokens during agent execution

### Finding Your Agent Identity

**For Shared Project ID:**
1. Go to [Azure Portal](https://portal.azure.com)
2. Navigate to your Foundry Project
3. Click **JSON View** in the Overview pane
4. Copy the `agentIdentityId` value

**For Individual Agent ID (after publishing):**
1. Navigate to the agent application resource
2. Click **JSON View** in the Overview pane
3. Copy the unique `agentIdentityId`

## Part 2: MCP Authentication Methods

### Choosing the Right Auth Method: Real-World Scenarios

Microsoft Foundry supports **4 main authentication methods** for MCP servers. Here's how to choose:

| Method | Enterprise Scenario | User Interaction | Example |
|--------|---------------------|------------------|---------|
| **Unauthenticated** | Public documentation, open APIs | None | Agent reads public Azure docs, weather APIs |
| **Key-based** | Internal APIs, SaaS integrations | None | Agent queries GitHub Enterprise, Jira, Confluence |
| **Agentic Identity** | Azure-native resources | None | Agent reads from Azure Blob Storage, queries Cosmos DB |
| **OAuth Passthrough** | User-specific access | Yes (consent) | Agent sends email *as user*, accesses *their* OneDrive |

### Real-World Enterprise Use Cases

| Use Case | Auth Method | Why |
|----------|-------------|-----|
| **IT Helpdesk Agent** queries ServiceNow tickets | Key-based | ServiceNow API key stored securely |
| **Sales Assistant** reads company CRM | Agentic Identity | Agent has its own CRM permissions |
| **Personal Scheduler** books meetings | OAuth Passthrough | Agent acts *as the user* in Outlook |
| **Code Assistant** searches Azure API specs | Unauthenticated | Public documentation |

### Decision Tree: Choosing MCP Auth Method

```
Does the MCP server require authentication?
│
├─ NO  → Use Unauthenticated
│
└─ YES → Does each user need their own identity/context?
         │
         ├─ YES → Use OAuth Identity Passthrough
         │        (User signs in, agent acts on their behalf)
         │
         └─ NO  → Is the resource an Azure service?
                  │
                  ├─ YES → Use Agentic Identity
                  │        (Assign RBAC to agent identity)
                  │
                  └─ NO  → Use Key-based
                           (Store API key in Project Connection)
```

## Understanding Azure AI APIs: Assistants vs Responses

Before diving into code, it's important to understand the **two API paradigms** available in Microsoft Foundry. This tutorial uses the newer **Responses API**, but you may encounter code using the older **Assistants API** pattern.

### API Evolution

| Feature | Assistants API (Preview) | Responses API (New) |
|---------|-------------------------|---------------------|
| **Status** | Preview (legacy pattern) | Generally Available |
| **State Management** | Threads + Messages + Runs | Conversations via `previous_response_id` |
| **Code Pattern** | `client.beta.threads.create()` | `openai_client.responses.create()` |
| **SDK** | `azure-ai-agents` (`McpTool`) | `azure-ai-projects` (`MCPTool`) |
| **MCP Support** | Yes | Yes with approval workflows |

### Threads vs Conversations: Key Differences

**Assistants API (Thread-based):**
```python
# 1. Create a thread (conversation container)
thread = client.beta.threads.create()

# 2. Add messages to the thread
client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="What's the weather?"
)

# 3. Run the assistant on the thread
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id
)

# 4. Poll for completion
while run.status != "completed":
    run = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)

# 5. Get messages
messages = client.beta.threads.messages.list(thread_id=thread.id)
```

**Responses API (Conversation-based):**
```python
# 1. Create initial response (starts a conversation)
response = openai_client.responses.create(
    model="gpt-4o",
    input="What's the weather?",
    tools=[mcp_tool]
)

# 2. Continue conversation with previous_response_id
follow_up = openai_client.responses.create(
    model="gpt-4o",
    input="What about tomorrow?",
    previous_response_id=response.id  # Links to previous context
)

# 3. Access output directly
print(response.output_text)
```

### Multi-User Application Design

When building enterprise applications that serve multiple users, consider:

| Aspect | Assistants API | Responses API |
|--------|----------------|---------------|
| **User Isolation** | Create separate `thread_id` per user | Use separate `previous_response_id` chains per user |
| **Session Storage** | Store `thread_id` in user session/DB | Store last `response.id` in user session/DB |
| **Cleanup** | Delete threads when done | Responses auto-expire (default 30 days) |
| **Concurrency** | Thread locked during run | No locking, async-friendly |

### Enterprise Architecture Pattern

```
┌─────────────────────────────────────────────────────────────┐
│                     Your Application                        │
├─────────────────────────────────────────────────────────────┤
│  User Session Storage (Redis/Database)                      │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐       │
│  │ User A       │  │ User B       │  │ User C       │       │
│  │ response_id: │  │ response_id: │  │ response_id: │       │
│  │ "resp_abc"   │  │ "resp_xyz"   │  │ "resp_123"   │       │
│  └──────────────┘  └──────────────┘  └──────────────┘       │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│              Microsoft Foundry (Responses API)               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  responses.create(previous_response_id=user_resp_id) │   │
│  │  → Each user maintains a separate conversation chain  │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

### Key Takeaways for This Tutorial

1. **Use the Responses API** - Modern GA approach with simpler code
2. **`previous_response_id`** - Chain responses to maintain conversation context
3. **`response.output_text`** - Aggregated text output from the model
4. **MCP Tool Approvals** - Responses API supports approval workflows for MCP tool calls
5. **No thread management** - No need to create/poll/manage threads

### SDK Reference

| Package | Import Pattern | API Style |
|---------|----------------|---------|
| `azure-ai-projects>=2.0.0b1` | `from azure.ai.projects.models import MCPTool` | Responses API |
| `azure-ai-agents>=1.2.0b3` | `from azure.ai.agents.models import McpTool` | Assistants API |

> **Note:** You can install both SDKs together, but be careful with import statements as the tool class names are similar but differ in casing (`MCPTool` vs `McpTool`).

## Part 3: Setup and Installation

Let's install the required packages for working with Microsoft Foundry agents and MCP.

In [ ]:
# Install required packages
# azure-ai-projects: Main SDK for Foundry projects and agents
# azure-ai-agents: Agent-specific features with MCP support
# azure-identity: Azure authentication (DefaultAzureCredential)

!pip install "azure-ai-projects>=2.0.0b1" "azure-ai-agents>=1.2.0b3" azure-identity python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# =============================================================================
# Configuration
# =============================================================================

# Microsoft Foundry Project Configuration
# Obtain from the Overview page of your Foundry project
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")

# Model deployment name from the "Models + endpoints" tab in your project
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

# Optional: MCP Project Connection ID (for key-based auth)
MCP_PROJECT_CONNECTION_ID = os.getenv("MCP_PROJECT_CONNECTION_ID", "")

# =============================================================================
# Your Custom MCP Server (from Tutorial 16 - ACA Deployment)
# =============================================================================

# Direct ACA endpoint (from Tutorial 16)
MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", "")
MCP_SERVER_NAME = os.getenv("MCP_SERVER_NAME", "travel-mcp-server")
MCP_BACKEND_URL = os.getenv("MCP_BACKEND_URL", "")

# =============================================================================
# APIM Integration (from Tutorial 18)
# =============================================================================

APIM_NAME = os.getenv("APIM_NAME", "")
APIM_RESOURCE_GROUP = os.getenv("APIM_RESOURCE_GROUP", "")
APIM_GATEWAY_URL = os.getenv("APIM_GATEWAY_URL", "")
APIM_SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY", "")

# Construct APIM-proxied MCP URL (recommended for production)
APIM_MCP_URL = f"{APIM_GATEWAY_URL}/travel-mcp/mcp" if APIM_GATEWAY_URL else ""

# =============================================================================
# Validation
# =============================================================================

def validate_config():
    """Validate required configuration."""
    print("📋 Configuration Status:")
    print("=" * 60)
    
    config_ok = True
    
    # Foundry Configuration
    print("\n🔹 Microsoft Foundry Configuration:")
    if PROJECT_ENDPOINT:
        print(f"  ✅ PROJECT_ENDPOINT: {PROJECT_ENDPOINT[:50]}...")
    else:
        print("  ❌ PROJECT_ENDPOINT: NOT SET")
        config_ok = False
    
    print(f"  ✅ MODEL_DEPLOYMENT_NAME: {MODEL_DEPLOYMENT_NAME}")
    
    if MCP_PROJECT_CONNECTION_ID:
        print(f"  ✅ MCP_PROJECT_CONNECTION_ID: {MCP_PROJECT_CONNECTION_ID[:30]}...")
    else:
        print("  ⚪ MCP_PROJECT_CONNECTION_ID: NOT SET (optional)")
    
    # Custom MCP Server (ACA)
    print("\n🔹 Custom MCP Server (Azure Container Apps):")
    if MCP_SERVER_URL:
        print(f"  ✅ MCP_SERVER_URL: {MCP_SERVER_URL}")
    else:
        print("  ⚪ MCP_SERVER_URL: NOT SET (see Tutorial 16)")
    
    if MCP_BACKEND_URL:
        print(f"  ✅ MCP_BACKEND_URL: {MCP_BACKEND_URL}")
    else:
        print("  ⚪ MCP_BACKEND_URL: NOT SET")
    
    # APIM Configuration
    print("\n🔹 API Management Integration:")
    if APIM_GATEWAY_URL:
        print(f"  ✅ APIM_GATEWAY_URL: {APIM_GATEWAY_URL}")
    else:
        print("  ⚪ APIM_GATEWAY_URL: NOT SET (see Tutorial 18)")
    
    if APIM_SUBSCRIPTION_KEY:
        print(f"  ✅ APIM_SUBSCRIPTION_KEY: {APIM_SUBSCRIPTION_KEY[:8]}...")
    else:
        print("  ⚪ APIM_SUBSCRIPTION_KEY: NOT SET")
    
    if APIM_MCP_URL:
        print(f"  ✅ APIM_MCP_URL: {APIM_MCP_URL}")
    
    print("\n" + "=" * 60)
    
    if not config_ok:
        print("\n📝 Required - Add to your .env file:")
        print('   AZURE_AI_PROJECT_ENDPOINT=https://your-project.api.azureml.ms')
        print('   AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o')
    
    return config_ok

CONFIG_OK = validate_config()

## Part 4: Unauthenticated MCP Server

### When to Use: Public Knowledge Sources

Unauthenticated MCP servers are ideal for **publicly available information** that doesn't require credentials.

**Enterprise Use Cases:**
- 📖 **Documentation Lookup**: Agent answers questions using official Azure/AWS/GCP docs
- 📊 **Public Data**: Weather, stock prices, exchange rates
- 🔍 **Open Source Exploration**: Agent reads public GitHub repositories

**Example MCP Servers (Public):**

| Server | URL | What the Agent Can Do |
|--------|-----|------------------------|
| Azure API Specs | `https://gitmcp.io/Azure/azure-rest-api-specs` | Look up Azure REST API definitions |
| Microsoft Learn | `https://learn.microsoft.com/api/mcp` | Search Microsoft documentation |

> 💡 **Tip**: Even for public servers, start with `require_approval="always"` to see what tools are being called before you fully trust them.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

def demo_unauthenticated_mcp():
    """
    Demonstrates using an unauthenticated (public) MCP server.
    
    This example connects to the GitHub MCP server for
    Azure REST API specifications, which is publicly accessible
    without authentication.
    """
    
    # Create MCP tool - no authentication needed
    mcp_tool = MCPTool(
        server_label="azure_api_specs",  # Unique label for this MCP server
        server_url="https://gitmcp.io/Azure/azure-rest-api-specs",  # Public MCP endpoint
        require_approval="always",  # Options: "always", "never", or specific tools
    )
    
    print(f"✅ Created MCP Tool:")
    print(f"   Label: {mcp_tool.server_label}")
    print(f"   URL: {mcp_tool.server_url}")
    print(f"   Approval: {mcp_tool.require_approval}")
    
    return mcp_tool

# Run the demo
mcp_tool_public = demo_unauthenticated_mcp()

## Part 5: Key-Based Authentication with Project Connections

### When to Use: SaaS APIs and Internal Services

Most enterprise integrations require API keys — GitHub, Jira, Salesforce, ServiceNow, etc. **Project Connections** let you store these securely in Azure.

**Enterprise Use Cases:**

| Service | What the Agent Can Do |
|---------|------------------|
| **GitHub Enterprise** | Code search, issue creation, PR reviews |
| **Jira/ServiceNow** | Ticket creation, status updates, backlog queries |
| **Salesforce** | Customer data lookup, opportunity updates |
| **Confluence** | Internal wiki search, runbook retrieval |

### Setting Up a Project Connection (Step-by-Step)

1. Go to [Microsoft Foundry Portal](https://ai.azure.com)
2. Navigate to your project → **Management** → **Connected resources**
3. Click **+ New connection** → **Custom keys**
4. Configure:
   - **Name**: `github-enterprise-connection`
   - **Key**: `Authorization`
   - **Value**: `Bearer ghp_xxxxxxxxxxxx` (PAT token)

### Finding Your `MCP_PROJECT_CONNECTION_ID`

The `MCP_PROJECT_CONNECTION_ID` is the **Azure Resource ID** of your Project Connection. You need this to reference the connection in your MCPTool configuration.

**Method 1: From Foundry Portal (Easiest)**
1. Go to [ai.azure.com](https://ai.azure.com) → Your Project → **Management** → **Connected resources**
2. Click on the connection name
3. Look for the **Resource ID** or copy from the URL (format below)

**Method 2: From Azure Portal**
1. Go to [portal.azure.com](https://portal.azure.com)
2. Navigate to AI Foundry resource → **Projects** → Your Project
3. Under **Settings** → **Connections**, click on the connection
4. Click **JSON View** and copy the `id` field

**Method 3: Using Azure CLI**
```bash
# List all connections in your Foundry project
az resource list \
  --resource-group <your-rg> \
  --resource-type "Microsoft.CognitiveServices/accounts/projects/connections" \
  --query "[].{name:name, id:id}" -o table
```

**Resource ID Format:**
```
/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.CognitiveServices/accounts/<ai-services-account>/projects/<project-name>/connections/<connection-name>
```

**Example `.env` entry:**
```bash
MCP_PROJECT_CONNECTION_ID=/subscriptions/12345678-1234-1234-1234-123456789abc/resourceGroups/my-rg/providers/Microsoft.CognitiveServices/accounts/my-ai-services/projects/my-project/connections/github-mcp-connection
```

### Why Project Connections?

| Approach | Problem |
|----------|------|
| ❌ Hardcoded secrets | Checked into git, visible in logs, security nightmare |
| ❌ Environment variables | Better, but visible to anyone with server access |
| ✅ **Project Connections** | Encrypted at rest, access-controlled, auditable, rotatable |

> 🔐 **Security Best Practice**: Project Connections use Azure Key Vault behind the scenes. Secrets are never exposed in logs or agent responses.

In [ ]:
def create_mcp_tool_with_connection(connection_id: str) -> MCPTool:
    """
    Creates an MCP tool that uses a Project Connection for authentication.
    
    Project Connections securely store API keys/tokens and
    automatically inject them when the agent calls the MCP server.
    
    Args:
        connection_id: The resource ID of the Project Connection
                      (found in Foundry Portal → Settings → Connections)
    
    Returns:
        MCPTool configured with the connection
    """
    
    mcp_tool = MCPTool(
        server_label="github-api",
        server_url="https://api.githubcopilot.com/mcp",  # GitHub's MCP endpoint
        require_approval="always",
        project_connection_id=connection_id,  # Reference to stored credentials
    )
    
    print(f"✅ Created MCP Tool with Project Connection:")
    print(f"   Label: {mcp_tool.server_label}")
    print(f"   URL: {mcp_tool.server_url}")
    print(f"   Connection ID: {connection_id[:40]}...")
    
    return mcp_tool

# Example usage (only if connection ID is configured)
if MCP_PROJECT_CONNECTION_ID:
    mcp_tool_keybased = create_mcp_tool_with_connection(MCP_PROJECT_CONNECTION_ID)
else:
    print("⚪ Skipping key-based demo - MCP_PROJECT_CONNECTION_ID not set")
    print("   Set up a Project Connection in Foundry Portal to enable this feature")

## Part 6: Agentic Identity Authentication

### When to Use: Azure-Native Resources

When your agent needs to access **Azure services** (Storage, Cosmos DB, Key Vault, Logic Apps), Agentic Identity is the cleanest approach — no API keys to manage!

**Enterprise Use Cases:**

| Scenario | Azure Resource | Agentic Identity Benefit |
|----------|---------------|---------------------------|
| **Document Agent** reads contracts | Azure Blob Storage | No keys — agent uses its own identity |
| **Data Agent** queries customer records | Azure Cosmos DB | RBAC controls exactly what the agent can access |
| **Workflow Agent** triggers automation | Azure Logic Apps | Agent has its own managed identity |
| **Secret Agent** retrieves configuration | Azure Key Vault | Fine-grained secret access per agent |

### How Agentic Identity Works

```
┌─────────────┐    1. Request token         ┌──────────────────┐
│   Foundry   │──────────────────────▶│    Entra ID      │
│   Agent     │                        │   (Agent's MI)   │
└─────────────┘◀──────────────────────└──────────────────┘
       │           2. Token (scoped)                   
       │                                       
       │       3. Call MCP with token          
       ▼                                       
┌────────────────────────────────────────────┐
│  Your MCP Server (on Azure)                │
│  • Validates JWT token                     │
│  • Checks RBAC permissions                 │
│  • Executes tool if authorized             │
└────────────────────────────────────────────┘
```

### Setup: Granting Agent Access to Azure Resources

**Step 1: Get Agent Identity ID** (the agent's service principal)
```bash
# In Azure Portal → Your Foundry Project → JSON View
# Look for: "agentIdentityId": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
```

**Step 2: Assign RBAC to Agent Identity**
```bash
# Example: Let agent read blobs (but NOT delete!)
az role assignment create \
  --assignee <agent-identity-id> \
  --role "Storage Blob Data Reader" \
  --scope /subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<account>
```

> 🔐 **Principle of Least Privilege**: Only grant the minimum permissions needed. Use "Reader" roles when possible.

**Step 3: Create Connection with AgenticIdentityToken**

In [ ]:
# Example: Creating a connection for Agentic Identity (via REST API)
# This is typically done via Azure Portal or Azure CLI

AGENTIC_IDENTITY_CONNECTION_EXAMPLE = """
# REST API request to create an Agentic Identity connection
# PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/
#     providers/Microsoft.CognitiveServices/accounts/{account}/
#     projects/{project}/connections/{connection_name}?api-version=2024-10-01

{
    "name": "storage-mcp-agentic",
    "type": "CognitiveServices/accounts/projects/connections",
    "properties": {
        "authType": "AgenticIdentityToken",  # Use agent's identity
        "group": "ServicesAndApps",
        "category": "RemoteTool",
        "target": "https://your-mcp-server.azurecontainerapps.io/mcp",
        "isSharedToAll": true,
        "audience": "https://storage.azure.com",  # Target service scope
        "Credentials": {},
        "metadata": {
            "ApiType": "Azure"
        }
    }
}
"""

print("📋 Agentic Identity Connection Example:")
print(AGENTIC_IDENTITY_CONNECTION_EXAMPLE)
print("\n💡 Key Points:")
print("   • authType: 'AgenticIdentityToken' tells Foundry to use the agent's identity")
print("   • audience: The scope URI of the target service")
print("   • No credentials needed - identity is automatic!")

### Common Audience Values for Azure Services

| Azure Service | Audience URI |
|--------------|---------------|
| Azure Storage | `https://storage.azure.com` |
| Azure Logic Apps | `https://logic.azure.com` |
| Microsoft Foundry | `https://ai.azure.com` |
| Microsoft Graph | `https://graph.microsoft.com` |
| Azure Key Vault | `https://vault.azure.net` |
| Azure Cosmos DB | `https://cosmos.azure.com` |

## Part 7: Creating an Agent with MCP Tools

Let's put it all together and create a Foundry agent with MCP capabilities.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

def create_agent_with_mcp(
    project_endpoint: str,
    model_name: str,
    mcp_server_url: str,
    mcp_server_label: str,
    project_connection_id: str = None,
):
    """
    Creates a Microsoft Foundry agent with MCP tool capabilities.
    
    This demonstrates the recommended pattern from Azure SDK samples.
    
    Args:
        project_endpoint: Foundry project endpoint
        model_name: Model deployment name (e.g., 'gpt-4o')
        mcp_server_url: URL of the MCP server
        mcp_server_label: Unique label for this MCP server
        project_connection_id: Optional connection ID for authenticated MCP
    
    Returns:
        tuple: (agent, project_client) - caller should clean up the client
    """
    
    # Create MCP tool
    mcp_tool_kwargs = {
        "server_label": mcp_server_label,
        "server_url": mcp_server_url,
        "require_approval": "always",
    }
    
    if project_connection_id:
        mcp_tool_kwargs["project_connection_id"] = project_connection_id
    
    mcp_tool = MCPTool(**mcp_tool_kwargs)
    tools: list[Tool] = [mcp_tool]
    
    # Create client and agent (synchronous API)
    credential = DefaultAzureCredential()
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )
    
    # Create agent version with MCP tool
    agent = project_client.agents.create_version(
        agent_name="mcp-demo-agent",
        definition=PromptAgentDefinition(
            model=model_name,
            instructions="""You are a helpful agent that can assist users using MCP tools.

Use the available MCP tools to:
- Search for information
- Access external resources
- Answer questions using connected data sources

Always explain what tools you're using and why.""",
            tools=tools,
        ),
        description="Demo agent with MCP capabilities",
    )
    
    print(f"✅ Agent created:")
    print(f"   ID: {agent.id}")
    print(f"   Name: {agent.name}")
    print(f"   Version: {agent.version}")
    
    return agent, project_client

# Example usage (if configured)
if CONFIG_OK:
    print("🚀 Ready to create agent")
    print("\nExample:")
    print("""agent, client = create_agent_with_mcp(
    project_endpoint=PROJECT_ENDPOINT,
    model_name=MODEL_DEPLOYMENT_NAME,
    mcp_server_url="https://gitmcp.io/Azure/azure-rest-api-specs",
    mcp_server_label="azure-api-specs",
)

# Don't forget to clean up:
# client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
# client.close()""")
else:
    print("❌ Configure PROJECT_ENDPOINT first (see Part 3)")

## Part 8: Handling MCP Approval Workflows

When `require_approval` is set to `"always"` or includes specific tools, the agent will request approval before executing MCP tools.

### Approval Flow

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│    Agent     │────▶│   MCP Tool   │────▶│   Approval   │
│   Request    │     │   Invoked    │     │   Required   │
└──────────────┘     └──────────────┘     └──────┬───────┘
                                                  │
                                                  ▼
                                        ┌──────────────────┐
                                        │ mcp_approval_    │
                                        │ request in       │
                                        │ response.output  │
                                        └────────┬─────────┘
                                                 │
                              ┌──────────────────┼──────────────────┐
                              ▼                                     ▼
                    ┌──────────────────┐               ┌──────────────────┐
                    │  User Approves   │               │  User Denies     │
                    │  (approve=True)  │               │  (approve=False) │
                    └────────┬─────────┘               └────────┬─────────┘
                             │                                  │
                             ▼                                  ▼
                    ┌──────────────────┐               ┌──────────────────┐
                    │ Tool Executes    │               │ Agent Notified   │
                    │ Agent Continues  │               │ of Denial        │
                    └──────────────────┘               └──────────────────┘
```

In [ ]:
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

def process_mcp_approval_requests(response, auto_approve: bool = False):
    """
    Processes MCP approval requests from agent responses.
    
    Args:
        response: Response from openai_client.responses.create()
        auto_approve: If True, automatically approve all requests
                     If False, prompt user for each request
    
    Returns:
        List of approval responses to send back to the agent
    """
    
    approval_responses: ResponseInputParam = []
    
    for item in response.output:
        if item.type == "mcp_approval_request":
            print(f"\n🔔 MCP Approval Request:")
            print(f"   Request ID: {item.id}")
            print(f"   Server: {item.server_label}")
            
            if auto_approve:
                approved = True
                print(f"   ✅ Auto-approved")
            else:
                # In a real application, you would prompt the user
                user_input = input("   Approve? (y/n): ").lower().strip()
                approved = user_input in ['y', 'yes']
                print(f"   {'✅ Approved' if approved else '❌ Denied'}")
            
            approval_responses.append(
                McpApprovalResponse(
                    type="mcp_approval_response",
                    approve=approved,
                    approval_request_id=item.id,
                )
            )
    
    return approval_responses

# Example usage
print("📋 MCP Approval Handler defined")
print("\nUsage pattern:")
print("""
# Send the initial request
response = openai_client.responses.create(
    conversation=conversation.id,
    input="Search for Azure REST API information",
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

# Process approvals
approvals = process_mcp_approval_requests(response, auto_approve=True)

# Continue with approvals
if approvals:
    response = openai_client.responses.create(
        input=approvals,
        previous_response_id=response.id,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    )
""")

## Part 9: OAuth Identity Passthrough

### When to Use: Acting on Behalf of Users

Sometimes the agent needs to do something *as the user* — send an email from their account, access their personal files, or book a meeting on their calendar.

**Key Difference from Other Auth Methods:**
- Key-based: Agent uses a **shared service account** (e.g., `support@company.com`)
- OAuth Passthrough: Agent acts **as the logged-in user** (e.g., `john@company.com`)

**Enterprise Use Cases:**

| Scenario | Why OAuth Passthrough? |
|----------|------------------------|
| **Personal Assistant** sends email | Email appears "from" the user, not a bot account |
| **Meeting Scheduler** books rooms | Uses user's calendar permissions, respects delegation rules |
| **File Finder** searches OneDrive | Only accesses files *that user* has permission to view |
| **Code Reviewer** creates GitHub PRs | PR appears as created by the actual developer |

### How OAuth Passthrough Works

```
┌──────────────┐    1. User asks agent to       ┌─────────────────┐
│     User     │    "Send email to client"       │   Foundry Agent │
└──────────────┘─────────────────────────────────▶└─────────────────┘
                                                       │
                       2. Agent needs Outlook access    │
                          but has no token              ▼
┌──────────────┐    3. "Click here to         ┌─────────────────┐
│     User     │    authorize Outlook"        │  Consent Prompt │
└──────────────┘◀─────────────────────────────└─────────────────┘
       │
       │ 4. User signs in and grants permission
       ▼
┌──────────────────┐     5. Token stored       ┌─────────────────┐
│  Microsoft Login │───────────────────────▶│  Foundry Vault  │
└──────────────────┘                        └─────────────────┘
                                                      │
                       6. Agent sends email using      │
                          user's token                 ▼
                                              ┌─────────────────┐
                                              │  Outlook (M365) │
                                              │  Email sent as  │
                                              │  john@company   │
                                              └─────────────────┘
```

### Microsoft 365 MCP Servers with OAuth

Microsoft provides **pre-built MCP servers** for M365 services — no server deployment needed!

| MCP Server | Permission Scope | Enterprise Scenario |
|------------|-----------------|---------------------|
| Outlook Mail | `McpServers.Mail.All` | Agent drafts/sends emails as the user |
| Outlook Calendar | `McpServers.Calendar.All` | Agent schedules the user's meetings |
| Teams | `McpServers.Teams.All` | Agent posts to Teams channels |
| SharePoint/OneDrive | `McpServers.OneDriveSharepoint.All` | Agent searches/summarizes user's documents |
| Copilot Search | `McpServers.CopilotMCP.All` | Agent searches enterprise content |

> ⚠️ **User Privacy Consideration**: With OAuth Passthrough, the agent sees what the user sees. Design prompts carefully to avoid unnecessarily accessing sensitive personal data.

In [ ]:
# Example: Handling OAuth consent flow

def handle_oauth_consent_request(response):
    """
    Handles OAuth consent requests from MCP tools.
    
    When OAuth is required, the response will contain an
    'oauth_consent_request' with a URL for the user to sign in.
    """
    
    for item in response.output:
        if item.type == "oauth_consent_request":
            print(f"\n🔐 OAuth Consent Required:")
            print(f"   Server: {item.server_label}")
            print(f"   Consent URL: {item.consent_url}")
            print(f"\n   Please have the user visit this URL to authorize access.")
            print(f"   After authorization, re-run the agent request.")
            return item.consent_url
    
    return None

print("📋 OAuth Consent Handler defined")
print("\n💡 OAuth Passthrough is ideal for:")
print("   • Personal GitHub repository access")
print("   • Microsoft 365 calendar/email integration")
print("   • User-specific data that shouldn't be shared")

## Part 10: Complete Example - Multi-MCP Agent

### Scenario: Technical Documentation Assistant

Let's build an agent that helps developers find information from multiple sources — demonstrating how a single agent can use multiple MCP servers with different authentication methods.

**What This Agent Can Do:**
- 📖 Search Azure REST API specifications (unauthenticated MCP)
- 📚 Query Microsoft Learn documentation (unauthenticated MCP)

> 💡 This pattern is common in enterprises: one agent connects to multiple knowledge sources, each with its own authentication requirements.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

def run_agent_with_mcp_demo():
    """
    Complete demonstration of a Foundry agent using MCP tools.
    
    This example shows:
    1. Creating an agent with unauthenticated MCP servers
    2. Using the conversation approach (Foundry Portal pattern)
    3. Handling approval requests
    4. Processing responses
    
    Note: The agent is not deleted, so you can reuse it or view it in the Foundry Portal.
    """
    
    if not CONFIG_OK:
        print("❌ Please configure PROJECT_ENDPOINT first")
        return
    
    # MCP Tool 1: Public Azure API Specs (unauthenticated)
    # Note: server_label must match ^[a-zA-Z0-9_]+$ (use underscores, not hyphens)
    mcp_tool_azure = MCPTool(
        server_label="azure_api_specs",
        server_url="https://gitmcp.io/Azure/azure-rest-api-specs",
        require_approval="always",
    )
    
    # MCP Tool 2: Microsoft Learn Documentation (unauthenticated)
    mcp_tool_learn = MCPTool(
        server_label="microsoft_learn",
        server_url="https://learn.microsoft.com/api/mcp",
        require_approval="never",
    )
    
    tools: list[Tool] = [mcp_tool_azure, mcp_tool_learn]
    
    credential = DefaultAzureCredential()
    
    # Use synchronous context manager
    with (
        AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project_client,
        project_client.get_openai_client() as openai_client,
    ):
        # Create agent (agent_name can use hyphens)
        agent = project_client.agents.create_version(
            agent_name="multi-mcp-demo-agent",
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT_NAME,
                instructions="""You are a helpful technical assistant with access to:

1. Azure REST API specifications (via azure_api_specs MCP)
2. Microsoft Learn documentation (via microsoft_learn MCP)

Use these tools to answer technical questions about Azure services.
Always cite your sources and explain which tools you're using.""",
                tools=tools,
            ),
            description="Agent with multiple MCP servers",
        )
        
        print(f"✅ Agent created: {agent.name} (v{agent.version})")
        print(f"   💡 Agent persists in Foundry Portal - not auto-deleted")
        
        # Create conversation (Foundry Portal pattern)
        conversation = openai_client.conversations.create()
        print(f"✅ Conversation created: {conversation.id}")
        
        # Send query using the conversation approach
        print("\n📝 Sending query...")
        response = openai_client.responses.create(
            conversation=conversation.id,
            input="What is Azure Cosmos DB and how do I create a container using the REST API?",
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        )
        
        # Process approval requests
        approval_requests = []
        for item in response.output:
            if item.type == "mcp_approval_request":
                print(f"\n🔔 Approval needed for: {item.server_label}")
                approval_requests.append(
                    McpApprovalResponse(
                        type="mcp_approval_response",
                        approve=True,  # Auto-approve for demo
                        approval_request_id=item.id,
                    )
                )
        
        # If there were approval requests, continue the conversation
        if approval_requests:
            print(f"✅ Approved {len(approval_requests)} request(s)")
            response = openai_client.responses.create(
                input=approval_requests,
                previous_response_id=response.id,
                extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
            )
        
        # Print final response - use output_text property that aggregates all text
        print("\n📄 Agent Response:")
        print("-" * 60)
        print(response.output_text)
        print("-" * 60)
        
        # Agent is NOT deleted - can be reused or viewed in Foundry Portal
        print(f"\n💡 Agent '{agent.name}' (v{agent.version}) persists in your Foundry project.")
        print(f"   View it at: {PROJECT_ENDPOINT.replace('/api/projects/', '/agents/')}")
        
        return agent

# Run the demo
print("🚀 Multi-MCP Agent Demo")
print("="* 60)
if CONFIG_OK:
    print("Ready to run! Execute: run_agent_with_mcp_demo()")
else:
    print("❌ Configure PROJECT_ENDPOINT to run this demo")

In [ ]:
# Actually run the demo if configured
# This cell executes the full agent workflow with MCP tools

if CONFIG_OK:
    print("🚀 Running Multi-MCP Agent Demo...")
    print("=" * 60)
    run_agent_with_mcp_demo()
else:
    print("❌ Configure PROJECT_ENDPOINT to run this demo")

## Part 11: Custom MCP Server Integration

### Scenario: Enterprise Travel Booking Agent

You've built a custom MCP server for travel booking (Tutorial 16) and want to have a Foundry agent use it. This is the pattern for **any internal API** you want to expose to AI agents.

**Real-World Enterprise Parallels:**

| Your Travel MCP | Enterprise Equivalent |
|-------------------|----------------------|
| `search_flights` | Query internal inventory systems |
| `check_hotel_availability` | Check resource availability (rooms, equipment, slots) |
| `convert_currency` | Call financial/ERP APIs for pricing |

### Deployed MCP Server (From Tutorial 16)

| Tool | Function | Enterprise Pattern |
|------|--------------|-------------------|
| `search_flights` | Search available flights | Any search/query operation |
| `check_hotel_availability` | Find hotels by location/date | Resource availability checks |
| `convert_currency` | Currency conversion | Pricing/calculation APIs |

### Connection Options: Development vs Production

| Environment | Endpoint | Security Model |
|-------------|----------|----------------|
| **Development** | Direct ACA URL | Simpler for testing |
| **Production** | Via APIM Gateway | Rate limiting, subscription keys, logging |

### Architecture with APIM

```
┌─────────────────────────────────────────────────────────────────────┐
│  Production Architecture: Foundry Agent → APIM → ACA MCP Server     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌──────────────┐                                                   │
│  │ Foundry Agent│                                                   │
│  │ (with MCP)   │                                                   │
│  └──────┬───────┘                                                   │
│         │                                                           │
│         │ 1. MCP Request + Auth Header                              │
│         ▼                                                           │
│  ┌────────────────────────────────────────────────────────────────┐│
│  │ Azure API Management                                           ││
│  │ ┌────────────────────────────────────────────────────────────┐ ││
│  │ │ Inbound Policies:                                          │ ││
│  │ │ • validate-azure-ad-token (JWT validation)                 │ ││
│  │ │ • rate-limit-by-key                                        │ ││
│  │ │ • set-header (X-User-Id from token)                        │ ││
│  │ └────────────────────────────────────────────────────────────┘ ││
│  └────────────────────────────────────────────────────────────────┘│
│         │                                                           │
│         │ 2. Validated request                                      │
│         ▼                                                           │
│  ┌────────────────────────────────────────────────────────────────┐│
│  │ Azure Container Apps                                           ││
│  │ ┌────────────────────────────────────────────────────────────┐ ││
│  │ │ FastMCP Travel Server                                      │ ││
│  │ │ • search_flights()                                         │ ││
│  │ │ • check_hotel_availability()                               │ ││
│  │ │ • convert_currency()                                       │ ││
│  │ └────────────────────────────────────────────────────────────┘ ││
│  └────────────────────────────────────────────────────────────────┘│
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Part 11a: Create MCPTool pointing to YOUR deployed Travel MCP Server
from azure.ai.projects.models import MCPTool

def create_travel_mcp_tool(use_apim: bool = True) -> MCPTool:
    """
    Creates an MCPTool for the deployed Travel MCP Server.
    
    Args:
        use_apim: If True, route through APIM (recommended for production)
                  If False, connect directly to ACA endpoint
    
    Returns:
        MCPTool configured for the Travel MCP Server
    """
    
    if use_apim and APIM_MCP_URL:
        server_url = APIM_MCP_URL
        print(f"🔐 Using APIM-proxied endpoint (production mode)")
    elif MCP_SERVER_URL:
        server_url = MCP_SERVER_URL
        print(f"🔓 Using direct ACA endpoint (development mode)")
    else:
        raise ValueError("No MCP server URL configured. Deploy your server first (Tutorial 16)")
    
    mcp_tool = MCPTool(
        server_label="travel-mcp",
        server_url=server_url,
        require_approval="always",  # Approval needed for booking operations
    )
    
    print(f"✅ Created Travel MCP Tool:")
    print(f"   Label: {mcp_tool.server_label}")
    print(f"   URL: {server_url}")
    print(f"   Tools available: search_flights, check_hotel_availability, convert_currency")
    
    return mcp_tool

# Create the tool if MCP server is configured
if MCP_SERVER_URL or APIM_MCP_URL:
    travel_mcp_tool = create_travel_mcp_tool(use_apim=bool(APIM_MCP_URL))
else:
    print("⚠️ No MCP server configured. Complete Tutorial 16 first to deploy your Travel MCP Server.")

### Part 11b: Why APIM in Production?

In enterprise deployments, you typically don't expose internal APIs directly. **Azure API Management (APIM)** acts as a secure gateway.

**Why It Matters:**

| Concern | Without APIM | With APIM |
|---------|-------------|----------|
| **Rate Limiting** | Runaway agent could overwhelm API | Configurable throttling per agent/user |
| **Monitoring** | Limited visibility | Full request/response logging, analytics |
| **Security** | Directly exposed | WAF, IP filtering, JWT validation |
| **Versioning** | Difficult to manage | Route to different backends by version |
| **Cost Control** | Unlimited usage | Quota enforcement, usage tracking |

**Authentication Flow with APIM:**
```
Foundry Agent → APIM (validates subscription key) → Your MCP Server (ACA)
```

> **Current Limitation**: MCPTool doesn't natively pass custom headers like `Ocp-Apim-Subscription-Key`. For APIM integration, configure APIM to:
> - Allow traffic from Foundry's IP range without a subscription key, or
> - Use a policy that injects the key based on the request source

In [ ]:
# Part 11c: Create Agent with Travel MCP Server (using Foundry Portal pattern)
# Reference: Microsoft Foundry Portal sample code

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition, Tool

def create_travel_agent_foundry_pattern(use_apim: bool = False):
    """
    Creates an agent that uses a custom Travel MCP Server
    deployed on Azure Container Apps.
    
    This uses the Foundry Portal pattern:
    - AIProjectClient 
    - MCPTool (uppercase 'CP')
    - openai_client.conversations.create() + responses.create()
    
    Note: The agent is not deleted, so you can reuse it or view it in the Foundry Portal.
    
    Args:
        use_apim: If True, use APIM gateway. If False, direct ACA endpoint.
    """
    
    # Get MCP server URL 
    if use_apim and APIM_MCP_URL:
        mcp_url = APIM_MCP_URL
        print("🔐 Using APIM-proxied endpoint")
    elif MCP_SERVER_URL:
        mcp_url = MCP_SERVER_URL
        print("🔓 Using direct ACA endpoint (no APIM)")
    else:
        print("⚠️ No MCP server configured. Complete Tutorial 16 first.")
        return
    
    print("🚀 Creating Travel Agent with Custom MCP Server (Foundry Pattern)...")
    print("=" * 60)
    print(f"   MCP URL: {mcp_url}")
    
    # Initialize MCP tool using azure.ai.projects MCPTool
    # Note: server_label must match ^[a-zA-Z0-9_]+$ (underscores OK, no hyphens)
    mcp_tool = MCPTool(
        server_label="travel_mcp",
        server_url=mcp_url,
        require_approval="never",  # Auto-approve for demo
    )
    
    tools: list[Tool] = [mcp_tool]
    
    credential = DefaultAzureCredential()
    
    with AIProjectClient(
        endpoint=PROJECT_ENDPOINT,
        credential=credential,
    ) as project_client:
        
        # Get OpenAI client for conversations
        openai_client = project_client.get_openai_client()
        
        # Create agent version with MCP tool
        # Note: agent_name can use hyphens (unlike server_label!)
        agent = project_client.agents.create_version(
            agent_name="travel-mcp-agent",  # Hyphens OK here
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT_NAME,
                instructions="""You are a helpful travel assistant powered by a custom MCP server.
            
You have access to the following tools:
- search_flights: Search for available flights between cities
- check_hotel_availability: Find hotels with availability  
- convert_currency: Convert between different currencies

When helping users plan travel, use these tools to provide accurate information.
Always present results clearly and offer to help with next steps.""",
                tools=tools,
            ),
            description="Travel assistant using custom ACA-hosted MCP server",
        )
        
        print(f"✅ Agent created: {agent.name} (v{agent.version})")
        print(f"   ID: {agent.id}")
        print(f"   Model: {MODEL_DEPLOYMENT_NAME}")
        print(f"   💡 Agent persists in Foundry Portal - not auto-deleted")
        print("-" * 60)
        
        # Create conversation (Foundry Portal pattern)
        conversation = openai_client.conversations.create()
        print(f"✅ Conversation created: {conversation.id}")
        
        # Test query using conversations approach (from Foundry Portal)
        test_query = "Search for flights from Seattle to New York for 2 people on December 20th"
        print(f"\n👤 User: {test_query}")
        print("\n🔄 Processing with Travel MCP Server...")
        
        response = openai_client.responses.create(
            conversation=conversation.id,
            input=test_query,
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        )
        
        print("\n🤖 Assistant response:")
        print("-" * 40)
        print(response.output_text)
        
        # Agent is NOT deleted - can be reused or viewed in Foundry Portal
        print(f"\n💡 Agent '{agent.name}' (v{agent.version}) persists in your Foundry project.")
        print(f"   You can continue conversations or view it in the Foundry Portal.")
        
        return agent, conversation.id

# Run the demo - use direct ACA endpoint (APIM requires subscription key)
if MCP_SERVER_URL or APIM_MCP_URL:
    try:
        create_travel_agent_foundry_pattern(use_apim=False)
    except Exception as e:
        print(f"⚠️ Error: {e}")
        import traceback
        traceback.print_exc()
        print("\nThis may occur if:")
        print("- The MCP server is not running or accessible")
        print("- Network connectivity issues")
else:
    print("⚠️ Travel MCP Server not configured. Complete Tutorial 16 first.")

## Part 12: Enterprise Best Practices and Security Checklist

### Authentication Security (For Security Reviews)

| Requirement | How to Implement | Why It Matters |
|-------------|-----------------|----------------|
| **No hardcoded secrets** | Use Project Connections for all credentials | Prevents secrets in git, logs, or agent output |
| **Least privilege access** | Grant minimal RBAC roles to agent identity | Limits blast radius if agent is compromised |
| **Audit everything** | Enable Azure Monitor + Log Analytics | Required for SOC2, HIPAA compliance |
| **Token hygiene** | Rely on Foundry's automatic refresh | Prevents long-lived token theft |
| **Individual IDs** | Publish agents for production | Per-agent audit trails and revocation |

### Approval Workflows (For Compliance)

| Data Sensitivity | Approval Setting | Example |
|-----------------|------------------|---------|
| **Read-only public data** | `"never"` | Searching public documentation |
| **Internal data reads** | `"always"` (initially) | Querying CRM, HR systems |
| **Write operations** | `"always"` (always) | Creating tickets, sending emails |
| **Financial/PII data** | `"always"` + human review | Accessing payment info, employee records |

### Production Readiness Checklist

**Identity and Access:**
- [ ] Agent published with individual identity (not shared project ID)
- [ ] RBAC reviewed and approved by security team
- [ ] No overly permissive roles (avoid "Owner", "Contributor" where possible)

**Secret Management:**
- [ ] All API keys in Project Connections (not in code)
- [ ] Secret rotation schedule documented
- [ ] Emergency revocation procedure tested

**Monitoring and Compliance:**
- [ ] Azure Monitor diagnostic settings enabled
- [ ] Alerts set for unusual activity (high error rates, unexpected tools)
- [ ] Log retention meets compliance requirements (90 days, 1 year, etc.)

**User Experience:**
- [ ] Approval prompts are clear and actionable
- [ ] OAuth consent screens reviewed for appropriate scopes
- [ ] Error messages don't leak sensitive information

## Summary: What You Learned

### The 4 MCP Authentication Patterns (Decision Guide)

| When You Need To... | Use This Auth | Example |
|--------------------|---------------|--------|
| Access public APIs/docs | **Unauthenticated** | Azure API specs, public wikis |
| Call internal/SaaS APIs | **Key-based** | GitHub Enterprise, Jira, Salesforce |
| Access Azure resources | **Agentic Identity** | Blob Storage, Cosmos DB, Key Vault |
| Act as a user | **OAuth Passthrough** | Send email as user, access their OneDrive |

### Enterprise Architecture Pattern

```
┌─────────────┐     ┌───────────────┐     ┌──────────────┐     ┌─────────────┐
│   User/App  │────▶│ Foundry Agent │────▶│  MCP Server  │────▶│  Backend    │
│             │     │ (with auth)   │     │  (ACA/APIM)  │     │  Services   │
└─────────────┘     └───────────────┘     └──────────────┘     └─────────────┘
                           │                      │
                    Agent Identity          Rate Limiting
                    RBAC Controls           Monitoring
                    Audit Logging           API Versioning
```

### Key Security Principles

1. ✅ **Never hardcode secrets** → Use Project Connections
2. ✅ **Least privilege** → Grant minimal permissions to agent IDs
3. ✅ **Audit everything** → Enable Azure Monitor for compliance
4. ✅ **Approve sensitive operations** → Use `require_approval` for write operations
5. ✅ **Individual IDs in production** → Publish agents for unique identities

### Recommended Learning Path

| Tutorial | What You'll Learn |
|----------|-------------------|
| **15**: FastMCP Server Basics | Build your own MCP server from scratch |
| **16**: Deploy to ACA | Host your MCP server on Azure |
| **18**: APIM Integration | Add enterprise gateway (rate limiting, monitoring) |
| **19**: Orchestrating with MCP | Multi-turn conversations with MCP tools |
| **This Tutorial (19b)** | Secure authentication for all scenarios |

### Official Resources

| Resource | Description |
|----------|-------------|
| [MCP Authentication Docs](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/mcp-authentication?view=foundry) | Official guide for all authentication patterns |
| [Agent Identity Concepts](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/agent-identity?view=foundry) | Deep dive into Entra agent identities |
| [Azure AI Projects SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects) | Python SDK with samples |
| [Foundry Portal](https://ai.azure.com) | Manage agents, connections, and deployments |